# Build your first enchilada Block

A Block contains a model and an algorithm for advancing that model. **Wheel owns
its changing state.** By the end of this tutorial, you will have a working
sinusoid-amplitude sampler, a noise-only block, and checks that the blocks can
be reused safely across campaigns and observation domains.

We will cover:

1. Decide what the block models and which inputs it supports.
2. Separate fixed configuration from returned state.
3. Implement `sample` and construct a complete initial result.
4. Start at a known injection, run a campaign, and check the posterior.
5. Continue a run and choose a domain through Wheel.
6. Publish noise estimates through the same Block interface.

The signals are synthetic, with amplitudes in arbitrary units. This teaches the
exchange contract, not a physical LISA waveform or response model.

## Setup

From the repository root:

```sh
uv sync --extra examples
uv pip install matplotlib
uv run --no-sync jupyter lab examples/create_block.ipynb
```

Choose that environment's Python kernel and run the notebook top to bottom.
The computations need only enchilada and NumPy on Python 3.12 or newer. Matplotlib
is optional and used for the diagnostic figure. The main tutorial needs no WDM,
Eryn, GBGPU, or orbit package. Installing optional packages after `uv sync`, then
launching with `--no-sync`, preserves them.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass

import numpy as np

from enchilada import (
    Block,
    BlockResult,
    DataCovariance,
    L1Data,
    TranslatedCovariance,
    Wheel,
    transform,
)
from enchilada.testing import check_block

## 1. Understand the Block contract

A normal Python class qualifies as a `Block` when it has these two members; no
inheritance, decorator, or plugin registration is required.

| Member | When Wheel uses it | What it does |
| --- | --- | --- |
| `name` | Registration and ledger lookup | A unique, nonempty name within this Wheel. |
| `sample(conditional_residual, noise_covariance, current_block_result, *, rng)` | Once per block per cycle | Advance the supplied current result against the current inputs and return a complete new result. |

The application constructs a complete initial `BlockResult` and supplies it to
`wheel.add(block, initial_block_result=...)`. This argument is required and cannot
be `None`. Registration validates and adopts that result without calling model
code or consuming any Wheel RNG draws. Prior draws, injected values, or another
estimate are application choices, made before registration.

For block *i*, the conditional residual is

```text
observations − sum of every OTHER block's accepted TDI signal
```

Its own signal is still present. Evaluate this residual against your candidate
signal. Wheel performs subtraction between blocks; your block never subtracts
another block or adds its own old contribution back.

```text
application builds initial BlockResult
                 ↓
wheel.add(block, initial_block_result=...) → validate → ledger + residual + covariance
                                                            ↓
                       conditional residual + covariance + current result + rng
                                                            ↓
                                                         sample
                                                            ↓
                       complete BlockResult → validate → ledger + residual + covariance
```

Wheel passes independent snapshots and commits the RNG only with an accepted
result. A failed block call leaves that block's accepted state unchanged; earlier
successful blocks in the same cycle remain committed. Orbit resources are shared
and must be treated as immutable.


## 2. Specify the model before implementing the interface

Our model is `signal(t) = amplitude * sin(2*pi*frequency_hz*t)` on channel A.
Frequency is fixed. Amplitude has a Gaussian prior with configurable mean and
standard deviation. The likelihood is Gaussian with the supplied covariance.

The block accepts real time-domain data, channel A, and the
`fractional_frequency` convention. These are explicit tutorial choices. A real
source model should validate its own channel, observable, TDI, and orbit needs.

| Fixed on the instance | Returned in each BlockResult |
| --- | --- |
| Name, frequency, prior settings, proposal scale, proposals per call | `model_parameters`: current amplitude |
| Reusable immutable model resources, if needed | `sampler_state`: accumulated proposal and acceptance counts |
| No changing parameters or mutable campaign history | `metadata`: descriptive diagnostics |

A counter is not a physical parameter. Here it demonstrates state carried across
calls; a larger sampler may also need walkers, proposal adaptation, temperatures,
or an external sampler's RNG state. Wheel already owns the supplied NumPy RNG.

In [ ]:
num_time_samples = 256
sample_rate_hz = 1.0
frequency_hz = 0.03125
true_amplitude = 1.4
noise_standard_deviation = 0.5
sample_times_s = np.arange(num_time_samples) / sample_rate_hz
signal_basis = np.sin(2 * np.pi * frequency_hz * sample_times_s)

observed = L1Data(
    channel_data={
        "A": true_amplitude * signal_basis
        + np.random.default_rng(11).normal(
            0.0, noise_standard_deviation, num_time_samples
        )
    },
    sample_rate_hz=sample_rate_hz,
    channel_names=("A",),
    tdi_generation="2.0",
    physical_observable="fractional_frequency",
)
noise_covariance = DataCovariance.from_variance(
    observed, time_sample_variance=noise_standard_deviation**2
)
print(f"{observed.num_time_samples} samples; injected amplitude {true_amplitude}")

## 3. Choose what `sample` means for this model

We will use a small symmetric random-walk Metropolis sampler. Each call proposes
several amplitudes. For an accepted proposal, it advances to that amplitude; for
a rejected proposal, it retains the previous amplitude. Both cases return the
complete signal and state.

For this one-parameter Gaussian problem, the exact conditional is also Gaussian.
In real time coordinates, with basis `b`, conditional residual `r`, and covariance
`C`, its precision and mean are:

```text
precision = 1 / prior_std² + bᵀ C⁺ b
mean      = (prior_mean / prior_std² + bᵀ C⁺ r) / precision
```

`C⁺` is implemented by `noise_covariance.solve(...)`; it applies precision on the
retained subspace. Using the operator preserves correlations and statistical
exclusions, including those carried by a `TranslatedCovariance`. The proper prior
keeps the amplitude posterior well-defined even when the basis is excluded.
These ordinary dot products apply here because the block requests real time
coordinates; raw Fourier dot products need the appropriate real-coordinate weights.

The proposal is symmetric, so its acceptance probability is the smaller of one
and the posterior density ratio. Constants independent of amplitude, including
the covariance determinant, cancel in that ratio. A block sampling noise generally
needs the determinant as well.

This conditional could be drawn directly. We use Metropolis to teach continuation
and rejection handling, then check its output against the exact answer. Several
Metropolis steps are a transition targeting the conditional, not an independent
exact conditional draw.

## 4. Implement the complete block

Read the class in this order:

1. `__post_init__` checks fixed settings; the frozen dataclass helps prevent
   accidental campaign state on the instance.
2. `_basis` validates the input conventions; `_render` builds a complete result.
3. `sample` reads amplitude and counters from `current_block_result`, refreshes
   the conditional target, proposes new values, and returns the full next result.

A helper beginning with `_` is ordinary implementation code; Wheel only calls
`sample`. Application setup will supply the initial amplitude, rendered signal,
and counters separately. This entire class can be copied into your own module.

In [ ]:
@dataclass(frozen=True)
class SineAmplitudeBlock:
    name: str
    frequency_hz: float
    prior_mean: float = 0.0
    prior_std: float = 2.0
    proposal_std: float = 0.08
    steps_per_call: int = 8

    def __post_init__(self):
        if not self.name:
            raise ValueError("Choose a nonempty block name")
        if not np.isfinite(self.frequency_hz) or self.frequency_hz <= 0:
            raise ValueError("frequency_hz must be finite and positive")
        if not np.isfinite(self.prior_mean):
            raise ValueError("prior_mean must be finite")
        if (
            not np.isfinite([self.prior_std, self.proposal_std]).all()
            or min(self.prior_std, self.proposal_std) <= 0
        ):
            raise ValueError("Prior and proposal standard deviations must be positive")
        if (
            isinstance(self.steps_per_call, bool)
            or not isinstance(self.steps_per_call, int)
            or self.steps_per_call < 1
        ):
            raise ValueError("steps_per_call must be a positive integer")

    def _basis(self, conditional_residual: L1Data) -> np.ndarray:
        if conditional_residual.data_domain != "time":
            raise ValueError("Register this block with data_domain='time'")
        if conditional_residual.channel_names != ("A",):
            raise ValueError("This tutorial block models only channel A")
        if conditional_residual.physical_observable != "fractional_frequency":
            raise ValueError("This tutorial expects fractional_frequency observations")
        if self.frequency_hz >= conditional_residual.sample_rate_hz / 2:
            raise ValueError("frequency_hz must be below Nyquist")
        times_s = (
            conditional_residual.start_time_gps
            + np.arange(conditional_residual.num_time_samples)
            / conditional_residual.sample_rate_hz
        )
        return np.sin(2 * np.pi * self.frequency_hz * times_s)

    def _render(self, conditional_residual, amplitude, num_proposals, num_acceptances):
        return conditional_residual.block_result(
            tdi_signal_contribution={
                "A": amplitude * self._basis(conditional_residual)
            },
            model_parameters={"amplitude": float(amplitude)},
            sampler_state={
                "num_proposals": num_proposals,
                "num_acceptances": num_acceptances,
            },
            metadata={"sampler": "random-walk Metropolis"},
        )

    def sample(
        self,
        conditional_residual: L1Data,
        noise_covariance: DataCovariance | TranslatedCovariance | None,
        current_block_result: BlockResult,
        *,
        rng: np.random.Generator,
    ) -> BlockResult:
        basis = self._basis(conditional_residual)
        if noise_covariance is None or noise_covariance.data_domain != "time":
            raise ValueError("This sampler needs a covariance in the time domain")
        noise_covariance.check_compatible(conditional_residual)
        precision_basis = noise_covariance.solve({"A": basis})["A"]
        precision_data = noise_covariance.solve(conditional_residual.channel_data)["A"]
        posterior_precision = 1 / self.prior_std**2 + float(basis @ precision_basis)
        posterior_mean = (
            self.prior_mean / self.prior_std**2 + float(basis @ precision_data)
        ) / posterior_precision

        def log_target(amplitude):
            return -0.5 * posterior_precision * (amplitude - posterior_mean) ** 2

        amplitude = float(current_block_result.model_parameters["amplitude"])
        num_proposals = current_block_result.sampler_state["num_proposals"]
        num_acceptances = current_block_result.sampler_state["num_acceptances"]
        # Other blocks may have changed the residual or covariance. Refresh the
        # current density for this call instead of caching it between calls.
        current_log_target = log_target(amplitude)
        for _ in range(self.steps_per_call):
            proposed_amplitude = rng.normal(amplitude, self.proposal_std)
            proposed_log_target = log_target(proposed_amplitude)
            acceptance_probability = np.exp(
                min(0.0, proposed_log_target - current_log_target)
            )
            num_proposals += 1
            if rng.random() < acceptance_probability:
                amplitude = proposed_amplitude
                current_log_target = proposed_log_target
                num_acceptances += 1

        return self._render(
            conditional_residual, amplitude, num_proposals, num_acceptances
        )

### Why these result fields matter

`tdi_signal_contribution` is the sum of all sources modeled by this block, on
its input grid. `conditional_residual.block_result(...)` checks the channel keys,
shape, and real/complex convention. Wheel additionally checks finiteness and
copies the result before adopting it.

A rejected proposal still returns the retained amplitude and its full signal.
**Returning `tdi_signal_contribution=None` means the block now contributes zero**;
it is not shorthand for keeping the previous signal. The same complete-snapshot
rule applies to parameters and sampler state. Keep open files, live samplers, and
other objects that cannot be deep-copied outside these dictionaries.

Our signal block omits `noise_covariance`, which leaves the global covariance
unchanged because this block is not its owner.

## 5. Build and check the complete initial state

Our main campaign starts at the known injected amplitude. Build the full result
that `sample` expects: the signal rendered from that amplitude, its physical
parameter value, and initialized proposal and acceptance counters. Other samplers
may also need walkers, adaptation, or an external sampler's RNG state. The result
must contain all continuation state even when a sampler starts at the true values.

`check_block` requires `initial_block_result`, constructs a scratch Wheel with it,
and takes a few sampling steps. It checks the exchange mechanics; it cannot
establish posterior correctness or prove that arbitrary user code is stateless.
It does not select a domain, so supply the initial signal, observations, and
covariance in this block's supported domain.


In [ ]:
signal_block = SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz)
configuration_before = deepcopy(vars(signal_block))
assert isinstance(signal_block, Block)

initial_at_injection = observed.block_result(
    {"A": true_amplitude * signal_basis},
    model_parameters={"amplitude": true_amplitude},
    sampler_state={"num_proposals": 0, "num_acceptances": 0},
)
check_block(
    block_under_test=signal_block,
    observed_data=observed,
    initial_block_result=initial_at_injection,
    initial_noise_covariance=noise_covariance,
    num_cycles=3,
    random_seed=5,
)
assert vars(signal_block) == configuration_before
assert initial_at_injection.sampler_state == {"num_proposals": 0, "num_acceptances": 0}
print("Initial parameters:", initial_at_injection.model_parameters)
print("Initial sampler state:", initial_at_injection.sampler_state)

## 6. Register the injection as the campaign's starting estimate

Pass the complete result to `Wheel.add(initial_block_result=...)`. Wheel does not
call an initializer or draw a prior; registration consumes no Wheel RNG draws.
Our amplitude, rendered signal, and proposal counters describe one consistent
starting state. Model authors must check physical consistency and prior support;
Wheel can only check the exchange contract.

The initial signal uses the domain and WDM grid requested at registration, or the
observation grid if no domain is requested. Here both are time-domain. Wheel
validates, copies, and converts the result to its observation grid before adopting
it. Optional covariance can use its own grid metadata and follows the usual
conversion and ownership rules, including for a noise-only starting result.

The observations still contain the injection plus noise. The initial estimate
sets the ledger and residual; the declared prior stays the same and amplitude
continues to be sampled. This initializes one block; a full campaign checkpoint
would also need the other blocks' states and the Wheel RNG state.

A prior start is another application choice: draw an amplitude with a separate
setup RNG, render its signal, and initialize the same counters before calling
`add`. Such a helper belongs in application or model setup and is not a required
Block method. Keep the prior in the sampling target regardless of the start.


In [ ]:
wheel = Wheel(observed, initial_noise_covariance=noise_covariance, random_seed=23)
pristine_values = observed.channel_data["A"].copy()
wheel.add(signal_block, initial_block_result=initial_at_injection, data_domain="time")
assert wheel.ledger["signal"].model_parameters["amplitude"] == true_amplitude
assert wheel.ledger["signal"].sampler_state == {
    "num_proposals": 0,
    "num_acceptances": 0,
}
np.testing.assert_array_equal(
    wheel.contribution("signal")["A"], true_amplitude * signal_basis
)
np.testing.assert_array_equal(wheel.observed_data.channel_data["A"], pristine_values)
np.testing.assert_allclose(
    wheel.working_residual.channel_data["A"],
    pristine_values - true_amplitude * signal_basis,
)
print("Known-injection start checks passed")

## 7. Collect a posterior chain from the injection start

Continue with the initialized `wheel` above. The ledger stores the latest accepted
result, not a history of every draw. Use `on_cycle_complete` to collect the chain
outside the block. Its callback runs after every block in that cycle is committed.

Below, each stored amplitude follows eight internal Metropolis proposals. Starting
at the injection does not fix the amplitude: every cycle advances it under the
same posterior target. We discard an initial warm-up segment before comparing
with the analytic posterior.


In [ ]:
amplitude_chain = []


def collect_amplitude(cycle_index, campaign):
    amplitude_chain.append(campaign.ledger["signal"].model_parameters["amplitude"])


num_cycles = 1200
warmup_cycles = 200
wheel.run(num_cycles=num_cycles, on_cycle_complete=collect_amplitude)
posterior_samples = np.asarray(amplitude_chain[warmup_cycles:])
last_result = wheel.ledger["signal"]
acceptance_fraction = (
    last_result.sampler_state["num_acceptances"]
    / last_result.sampler_state["num_proposals"]
)
assert (
    last_result.sampler_state["num_proposals"]
    == num_cycles * signal_block.steps_per_call
)
assert vars(signal_block) == configuration_before
assert initial_at_injection.sampler_state == {"num_proposals": 0, "num_acceptances": 0}
print(
    f"Posterior amplitude: {posterior_samples.mean():.4f} ± {posterior_samples.std():.4f}"
)
print(f"Acceptance fraction: {acceptance_fraction:.3f}")

## 8. Check the model and sampler, not just the interface

The initial example uses known white noise. Independently compute the analytic
amplitude posterior from the raw arrays and its known variance, then compare the
sample mean and spread. These checks complement `check_block`.

For a new physical block, add injection/recovery checks, likelihood comparisons,
prior tests, convergence diagnostics, and checks of waveform conventions. The
small check here is appropriate for this one-parameter tutorial; it is not a
convergence guarantee for an arbitrary sampler.

In [ ]:
reference_precision = (
    1 / signal_block.prior_std**2
    + float(signal_basis @ signal_basis) / noise_standard_deviation**2
)
reference_mean = (
    signal_block.prior_mean / signal_block.prior_std**2
    + float(signal_basis @ observed.channel_data["A"]) / noise_standard_deviation**2
) / reference_precision
reference_std = np.sqrt(1 / reference_precision)

assert abs(posterior_samples.mean() - reference_mean) < 0.25 * reference_std
assert abs(posterior_samples.std() / reference_std - 1) < 0.20
assert abs(reference_mean - true_amplitude) < 5 * reference_std
np.testing.assert_allclose(
    wheel.working_residual.channel_data["A"],
    observed.channel_data["A"] - last_result.tdi_signal_contribution["A"],
)
print(f"Analytic amplitude: {reference_mean:.4f} ± {reference_std:.4f}")
print("Posterior checks passed")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is None:
    print("Optional plots: install matplotlib in this kernel to display them.")
else:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    axes[0].plot(amplitude_chain, linewidth=0.7)
    axes[0].axvline(warmup_cycles, color="grey", linestyle="--", label="End of warm-up")
    axes[0].axhline(reference_mean, color="black", linestyle=":", label="Analytic mean")
    axes[0].set(title="Amplitude chain", xlabel="Wheel cycle", ylabel="Amplitude")
    axes[0].legend()

    amplitude_grid = np.linspace(
        reference_mean - 4 * reference_std, reference_mean + 4 * reference_std, 200
    )
    reference_density = np.exp(
        -0.5 * ((amplitude_grid - reference_mean) / reference_std) ** 2
    ) / (reference_std * np.sqrt(2 * np.pi))
    axes[1].hist(
        posterior_samples, bins=30, density=True, alpha=0.65, label="Metropolis draws"
    )
    axes[1].plot(
        amplitude_grid, reference_density, color="black", label="Analytic posterior"
    )
    axes[1].axvline(
        true_amplitude, color="tab:red", linestyle="--", label="Injected value"
    )
    axes[1].set(title="Posterior check", xlabel="Amplitude", ylabel="Density")
    axes[1].legend()

    axes[2].plot(
        sample_times_s, observed.channel_data["A"], alpha=0.55, label="Observed A"
    )
    axes[2].plot(
        sample_times_s,
        posterior_samples.mean() * signal_basis,
        label="Posterior mean signal",
    )
    axes[2].set(
        title="Signal reconstruction", xlabel="Time from start [s]", ylabel="Amplitude"
    )
    axes[2].legend()
    plt.show()

## 9. Verify continuation and campaign isolation

The same configured instance can be registered in independent Wheels. With the
same seed, initial result, and inputs, running two then three cycles must match
a single five-cycle run. Wheel keeps the RNG and current result between `run`
calls. Callback cycle indices, however, restart at zero for each invocation.

Ledger lookups return independent copies. Editing a retrieved result cannot
change the campaign. Persistence to disk is a separate concern; this check is
in-memory continuation, not a checkpoint format.

In [ ]:
continued = Wheel(observed, noise_covariance, random_seed=31)
uninterrupted = Wheel(observed, noise_covariance, random_seed=31)
for campaign in (continued, uninterrupted):
    campaign.add(
        signal_block, initial_block_result=initial_at_injection, data_domain="time"
    )
continued.run(2)
continued.run(3)
uninterrupted.run(5)
continued_result = continued.ledger["signal"]
uninterrupted_result = uninterrupted.ledger["signal"]
assert continued_result.model_parameters == uninterrupted_result.model_parameters
assert continued_result.sampler_state == uninterrupted_result.sampler_state
np.testing.assert_array_equal(
    continued_result.tdi_signal_contribution["A"],
    uninterrupted_result.tdi_signal_contribution["A"],
)
continued_result.model_parameters["amplitude"] = 999.0
assert continued.ledger["signal"].model_parameters != continued_result.model_parameters
assert vars(signal_block) == configuration_before
print("Continuation and isolation checks passed")

## 10. Keep domain translation in Wheel

The same block can work when the campaign stores Fourier observations. Register
it with `data_domain="time"`; Wheel translates the conditional residual, covariance,
and previous signal for the call, then returns the accepted signal to the original
grid for storage. The initial signal uses the requested time-domain grid, even
though the observations are stored in frequency coordinates. Parameters and
sampler state keep their block-local meaning.

Our block never calls `transform`. The application selects the representation at
registration. The following example uses only NumPy and retains the same draws.

In [ ]:
spectrum = transform(observed, "frequency")
frequency_campaign = Wheel(
    spectrum,
    initial_noise_covariance=transform(noise_covariance, "frequency"),
    random_seed=31,
)
frequency_campaign.add(
    signal_block, initial_block_result=initial_at_injection, data_domain="time"
)
frequency_campaign.run(5)
np.testing.assert_allclose(
    frequency_campaign.ledger["signal"].model_parameters["amplitude"],
    uninterrupted.ledger["signal"].model_parameters["amplitude"],
    atol=1e-12,
)
assert frequency_campaign.working_residual.data_domain == "frequency"
assert (
    frequency_campaign.contribution("signal")["A"].shape
    == spectrum.channel_data["A"].shape
)
np.testing.assert_allclose(
    transform(frequency_campaign.working_residual, "time").channel_data["A"],
    uninterrupted.working_residual.channel_data["A"],
    atol=1e-12,
)


# A genuinely translated time covariance can also be consumed via solve.
# Excluding Fourier DC corresponds to a nonlocal statistical exclusion in time.
@dataclass(frozen=True)
class FlatPSD:
    level: float

    def psd(self, frequencies, channel=None):
        return np.full_like(frequencies, self.level)


native_spectral_noise = DataCovariance.from_psd(
    spectrum, FlatPSD(2 * noise_standard_deviation**2 / sample_rate_hz)
)
# from_psd constructs frequency covariance and excludes DC by convention.
masked_time_noise = transform(native_spectral_noise, "time")
assert isinstance(masked_time_noise, TranslatedCovariance)
check_block(
    signal_block,
    observed,
    initial_block_result=initial_at_injection,
    initial_noise_covariance=masked_time_noise,
    num_cycles=2,
)
print("Domain handoff checks passed")

For a WDM campaign, the application can instead create
`transform(observed, "wdm", num_frequency_divisions=8)` and register this same
block with `data_domain="time"`. A block whose own algorithm uses WDM would
request `data_domain="wdm"` and one division count. Both counts must be positive
and even, and their product must equal the number of underlying time samples.

WDM requires the separately installed local backend and Python 3.13 or newer.
The [domain translation notebook](domain_translation.ipynb) demonstrates that
setup and the two division selectors. Covariance in a new domain need not be
independent per pixel: use its operators or explicitly reject unsupported models.

## 11. A noise-only block uses the same interface

There is no separate NoiseBlock base class. A noise-only block returns
`BlockResult(noise_covariance=..., model_parameters=...)` with no signal array.

This example samples a shared white-noise variance with a proper inverse-gamma
prior. Given the conditional residual, its conjugate posterior has
`shape = prior_shape + number_of_samples/2` and
`scale = prior_scale + sum_of_squared_residuals/2`. These equations assume fully
observed real time samples and independent Gaussian noise. They do not model gaps
or correlated noise.

The draw is exact and independent of the previous variance, so `sampler_state`
can be empty. `model_parameters` records the physical variance; the resulting
`DataCovariance` tells other blocks how to weight their residuals. Initialize this
block with that full covariance, the corresponding physical variance, and an empty
sampler-state dictionary.

In [ ]:
@dataclass(frozen=True)
class WhiteNoiseVarianceBlock:
    name: str = "noise"
    prior_shape: float = 2.0
    prior_scale: float = 0.25

    def __post_init__(self):
        if (
            not np.isfinite([self.prior_shape, self.prior_scale]).all()
            or min(self.prior_shape, self.prior_scale) <= 0
        ):
            raise ValueError("Inverse-gamma prior shape and scale must be positive")

    def _result(self, conditional_residual, variance):
        if conditional_residual.data_domain != "time":
            raise ValueError("This white-noise model requires time-domain inputs")
        return BlockResult(
            noise_covariance=DataCovariance.from_variance(
                conditional_residual, variance
            ),
            model_parameters={"variance": float(variance)},
            sampler_state={},
        )

    def sample(
        self, conditional_residual, noise_covariance, current_block_result, *, rng
    ):
        if conditional_residual.data_domain != "time":
            raise ValueError("This white-noise model requires time-domain inputs")
        number_of_samples = sum(
            values.size for values in conditional_residual.channel_data.values()
        )
        squared_residual_sum = sum(
            float(values @ values)
            for values in conditional_residual.channel_data.values()
        )
        posterior_shape = self.prior_shape + 0.5 * number_of_samples
        posterior_scale = self.prior_scale + 0.5 * squared_residual_sum
        return self._result(
            conditional_residual, posterior_scale / rng.gamma(posterior_shape)
        )

In [ ]:
initial_noise_result = BlockResult(
    noise_covariance=noise_covariance,
    model_parameters={"variance": noise_standard_deviation**2},
    sampler_state={},
)
joint_campaign = Wheel(observed, random_seed=9)
joint_campaign.add(
    WhiteNoiseVarianceBlock(),
    initial_block_result=initial_noise_result,
    data_domain="time",
)
joint_campaign.add(
    signal_block, initial_block_result=initial_at_injection, data_domain="time"
)
joint_campaign.run(50)
noise_result = joint_campaign.ledger["noise"]
assert noise_result.tdi_signal_contribution is None
assert noise_result.sampler_state == {}
assert noise_result.noise_covariance is not None
np.testing.assert_array_equal(joint_campaign.contribution("noise")["A"], 0.0)
np.testing.assert_allclose(
    joint_campaign.noise_covariance.noise_variance("A"),
    noise_result.model_parameters["variance"],
)
print("Last noise variance draw:", noise_result.model_parameters["variance"])
print("Noise-only block checks passed")

The block that owns the current covariance must republish its complete covariance
on every call. Another block taking ownership replaces that model with a warning;
Wheel does not add noise models together. Publish combined noise components in one
covariance. A joint signal-and-noise block can fill both estimate fields in the
same `BlockResult`.

A block that only tracks state may return `BlockResult(sampler_state={"num_calls": 1})`.
That contributes no signal and, unless it owns the current covariance, leaves the
noise unchanged. Any previous signal from that block is removed.

## Before you plug in your own model

- Choose the supported channels, observable, TDI convention, domain, and orbit
  needs, then reject unsupported inputs clearly.
- Keep fixed settings and immutable resources on the block. Return every changing
  parameter and continuation value; make them safe to deep-copy.
- Implement `sample`; supply a complete, consistent `initial_block_result` to
  every `Wheel.add` and `check_block` call. Choose the start in application setup,
  including any optional prior draw, and include all state that `sample` needs.
- Recompute the conditional likelihood when the residual or noise changes.
- Return a full result after acceptance **and rejection**. Sum all sources modeled
  by your block before publishing its TDI signal.
- Use covariance operators for correlated or translated models. Include the
  determinant when the proposed parameters change the noise normalization.
- Run `check_block`, then independent numerical, posterior, and campaign-isolation
  checks. Collect chains outside the block through a callback.
- Let Wheel handle cross-block residuals and domain conversion.

The [toy fit script](toy_fit.py) provides a compact conjugate sampler, while the
[translation tutorial](domain_translation.ipynb) covers WDM layouts and covariance
semantics in more detail.